In [1]:
# CELLULE 1 - Setup environnement LOCAL (CPU) - Semaine 3 : préparation des données DPO

import subprocess, sys, platform

def run(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=False)

print("Machine :", platform.platform())
print("Python  :", sys.version)

packages = [
    "datasets",
    "huggingface_hub",
    "pandas",
]

run(f"{sys.executable} -m pip install -q " + " ".join(packages))

print("\n✅ Setup terminé (mode CPU - préparation des données DPO).")
import datasets, pandas as pd
print("datasets:", datasets.__version__)
print("pandas:", pd.__version__)

Machine : macOS-10.16-x86_64-i386-64bit
Python  : 3.11.13 (main, Jun  5 2025, 08:14:07) [Clang 14.0.6 ]
$ /opt/anaconda3/envs/triage-poc/bin/python -m pip install -q datasets huggingface_hub pandas

✅ Setup terminé (mode CPU - préparation des données DPO).
datasets: 5.0.1
pandas: 3.0.5


In [3]:
# CELLULE 2 - Préparation de l'échantillon DPO (hors chevauchement avec le SFT)

from datasets import load_dataset
import pandas as pd

# Recharger le dataset complet UltraMedical-Preference
ultramed_full = load_dataset("TsinghuaC3I/UltraMedical-Preference")
print(f"Total disponible : train={len(ultramed_full['train'])}, val={len(ultramed_full['validation'])}, test={len(ultramed_full['test'])}")

df_ultramed_train = ultramed_full["train"].to_pandas()

# Exclure les prompt_id déjà utilisés dans le SFT (les 800 échantillonnés avec random_state=42)
echantillon_sft_utilise = df_ultramed_train.sample(n=800, random_state=42)
ids_deja_utilises = set(echantillon_sft_utilise["prompt_id"])
print(f"IDs déjà utilisés en SFT : {len(ids_deja_utilises)}")

df_ultramed_disponible = df_ultramed_train[~df_ultramed_train["prompt_id"].isin(ids_deja_utilises)]
print(f"Paires disponibles pour le DPO (hors chevauchement SFT) : {len(df_ultramed_disponible)}")

# Échantillonnage pour le DPO - taille raisonnable pour un POC
N_ECHANTILLON_DPO = 1500
echantillon_dpo = df_ultramed_disponible.sample(n=N_ECHANTILLON_DPO, random_state=123).reset_index(drop=True)

print(f"\n✅ Échantillon DPO sélectionné : {len(echantillon_dpo)} paires")
print("\nColonnes disponibles :", echantillon_dpo.columns.tolist())
print("\nExemple :")
print(f"Prompt : {echantillon_dpo.iloc[0]['prompt'][:200]}")
print(f"\nLabel type : {echantillon_dpo.iloc[0]['label_type']}")
print(f"Score chosen : {echantillon_dpo.iloc[0]['metadata']['chosen']['score']}")
print(f"Score rejected : {echantillon_dpo.iloc[0]['metadata']['rejected']['score']}")

Total disponible : train=109353, val=2232, test=777
IDs déjà utilisés en SFT : 798
Paires disponibles pour le DPO (hors chevauchement SFT) : 107958

✅ Échantillon DPO sélectionné : 1500 paires

Colonnes disponibles : ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback']

Exemple :
Prompt : sir,I am NIDDM and sugar is under control. During 2004 TMT result was + for Ischeamia, during 2006 & 2008 it was -ve for inducible ischeamia and now again it is + for Ischeamia (test was terminated du

Label type : length
Score chosen : 4.0
Score rejected : 3.0


In [5]:
# CELLULE 3 - Inspection des types de labels de préférence

print("Distribution des label_type dans l'échantillon DPO :")
print(echantillon_dpo["label_type"].value_counts())

print("\nDistribution des label_type dans TOUT le dataset (pour référence) :")
print(df_ultramed_disponible["label_type"].value_counts())

# Vérifier l'écart de score moyen entre chosen et rejected par label_type
def extraire_scores(row):
    try:
        return pd.Series({
            "score_chosen": row["metadata"]["chosen"]["score"],
            "score_rejected": row["metadata"]["rejected"]["score"],
        })
    except (TypeError, KeyError):
        return pd.Series({"score_chosen": None, "score_rejected": None})

scores_df = echantillon_dpo.apply(extraire_scores, axis=1)
echantillon_dpo_scores = pd.concat([echantillon_dpo, scores_df], axis=1)

echantillon_dpo_scores["ecart_score"] = echantillon_dpo_scores["score_chosen"] - echantillon_dpo_scores["score_rejected"]

print("\nÉcart de score moyen (chosen - rejected) par label_type :")
print(echantillon_dpo_scores.groupby("label_type")["ecart_score"].agg(["mean", "count"]))

Distribution des label_type dans l'échantillon DPO :
label_type
hard      648
length    531
easy      321
Name: count, dtype: int64

Distribution des label_type dans TOUT le dataset (pour référence) :
label_type
hard      45229
length    40538
easy      22191
Name: count, dtype: int64

Écart de score moyen (chosen - rejected) par label_type :
                mean  count
label_type                 
easy        1.289720    321
hard        0.975309    648
length      1.450094    531


In [7]:
# CELLULE 4 - Formatage final pour DPOTrainer (extraction chosen/rejected en texte simple)

def extraire_reponse_assistant(messages):
    """Extrait le contenu du message assistant depuis la liste de messages chosen/rejected."""
    for msg in messages:
        if msg.get("role") == "assistant":
            return msg.get("content", "")
    return ""

lignes_dpo = []

for _, row in echantillon_dpo.iterrows():
    prompt = row["prompt"]
    chosen_texte = extraire_reponse_assistant(row["chosen"])
    rejected_texte = extraire_reponse_assistant(row["rejected"])

    if not prompt or not chosen_texte or not rejected_texte:
        continue

    lignes_dpo.append({
        "prompt_id": row["prompt_id"],
        "label_type": row["label_type"],
        "prompt": prompt,
        "chosen": chosen_texte,
        "rejected": rejected_texte,
    })

df_dpo_final = pd.DataFrame(lignes_dpo)
print(f"✅ {len(df_dpo_final)} paires formatées pour le DPO (sur {len(echantillon_dpo)} initiales)")

print("\nAperçu :")
print(f"Prompt   : {df_dpo_final.iloc[0]['prompt'][:150]}")
print(f"Chosen   : {df_dpo_final.iloc[0]['chosen'][:150]}")
print(f"Rejected : {df_dpo_final.iloc[0]['rejected'][:150]}")

# Sauvegarde
import os
os.makedirs("data_processed", exist_ok=True)
df_dpo_final.to_csv("data_processed/dataset_dpo.csv", index=False)
print("\n💾 Sauvegardé -> data_processed/dataset_dpo.csv")

✅ 1500 paires formatées pour le DPO (sur 1500 initiales)

Aperçu :
Prompt   : sir,I am NIDDM and sugar is under control. During 2004 TMT result was + for Ischeamia, during 2006 & 2008 it was -ve for inducible ischeamia and now a
Chosen   : 

I'd be happy to help you understand the TMT results!

**What is a TMT (Treadmill Test)?**

A Treadmill Test, also known as an Exercise Stress Test, 
Rejected : I'd be happy to help you understand the TMT (Treadmill Stress Test) results and what they might mean for your condition.

The TMT is a non-invasive te

💾 Sauvegardé -> data_processed/dataset_dpo.csv


In [9]:
# CELLULE 5 - Split train/val + upload HF Hub

import pandas as pd
from sklearn.model_selection import train_test_split

df_dpo_final = pd.read_csv("data_processed/dataset_dpo.csv")

# Split 90% train / 10% val (le DPO a généralement moins besoin d'un gros val set)
train_dpo, val_dpo = train_test_split(
    df_dpo_final, test_size=0.1, random_state=42, stratify=df_dpo_final["label_type"]
)

print(f"Train : {len(train_dpo)} ({len(train_dpo)/len(df_dpo_final)*100:.1f}%)")
print(f"Val   : {len(val_dpo)} ({len(val_dpo)/len(df_dpo_final)*100:.1f}%)")

print("\nRépartition label_type par split :")
print(pd.crosstab(pd.concat([train_dpo, val_dpo])["label_type"],
                   pd.concat([train_dpo.assign(split="train"), val_dpo.assign(split="val")])["split"]))

train_dpo.to_csv("data_processed/dpo_train.csv", index=False)
val_dpo.to_csv("data_processed/dpo_val.csv", index=False)
print("\n💾 Splits sauvegardés localement")

Train : 1350 (90.0%)
Val   : 150 (10.0%)

Répartition label_type par split :
split       train  val
label_type            
easy          289   32
hard          583   65
length        478   53

💾 Splits sauvegardés localement


In [11]:
# CELLULE 6 - Upload sur Hugging Face

from huggingface_hub import login
import getpass
from datasets import Dataset, DatasetDict

token = getpass.getpass("Token Hugging Face (write) : ")
login(token=token)

train_ds = Dataset.from_pandas(train_dpo.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_dpo.reset_index(drop=True))

dataset_dpo_dict = DatasetDict({"train": train_ds, "validation": val_ds})

repo_id = "UserMarrakech/chsa-triage-dpo"
dataset_dpo_dict.push_to_hub(repo_id, private=True)

print(f"\n✅ Dataset DPO uploadé sur https://huggingface.co/datasets/{repo_id}")

Token Hugging Face (write) :  ········


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✅ Dataset DPO uploadé sur https://huggingface.co/datasets/UserMarrakech/chsa-triage-dpo
